In [1]:
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np

#from datetime import datetime
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print('Error:',e)

Conexion a la base de datos realizada


In [11]:
with engine.connect() as connection:
    try:

            query = f"""
        SELECT DISTINCT B.IDENTIFICACION, B.PERIODO as 'ULTIMO PERIODO CURSADO', D.EQUIVALENCIA_MEN_AÑO AS 'AÑO ULTIMO PERIODO CURSADO',D.EQUIVALENCIA_MEN_SEMESTRE AS 'SEMESTRE ULTIMO PERIODO CURSADO' ,
        B.CICLO, B.Semestre_SINU, B.AÑO_GRAD_OPORTUNA, B.SEMESTRE_MEN_GRAD_OPORTUNA, B.programa, B.Tipo_grado_corregido AS 'ESTADO_GRADO',
        CASE WHEN A.GRADO_TECNICO IS NULL THEN 'NO' ELSE A.GRADO_TECNICO END AS GRADO_TECNICO,
        CASE WHEN A.GRADO_TECNOLOGO IS NULL THEN 'NO' ELSE A.GRADO_TECNOLOGO END AS GRADO_TECNOLOGO,
        CASE WHEN A.GRADO_PROFESIONAL IS NULL THEN 'NO' ELSE A.GRADO_PROFESIONAL END AS GRADO_PROFESIONAL,
        C.TOTAL_CREDITOS_NIVEL, C.creditos_cursados, C.creditos_aprobados, C.promedio_periodo, C.porcentaje_aprobacion
        FROM SRUBIO.GRADO_OPORTUNO2_TOTAL B
        LEFT JOIN GRADOS.TIENE_GRADO A
        ON B.Identificacion = A.NUMERO
        LEFT JOIN DESERCION.PORCENTAJE_APROBACION_POR_CICLO C
        ON B.IDENTIFICACION = C.IDENTIFICACION 
        AND B.CICLO = C.NIVEL
        LEFT JOIN ANALITICA.EQUIVALENCIA_AÑO_SEMESTRE_PERIODOS D
        ON B.Periodo = D.PERIODO
        WHERE Tipo_grado_corregido != 'Aun no se debe hacer seguimiento'    
        
            """
            df = pd.read_sql(query, engine)
            # Agregar el resultado a la lista si no está vacío

    finally:
        
        connection.close()

In [12]:
df_no_graduados = df.copy()

In [ ]:
def resumen_grados(df):
    """
    Recibe un DataFrame con columnas:
    IDENTIFICACION, GRADO_TECNICO, GRADO_TECNOLOGO, GRADO_PROFESIONAL
    Devuelve un DataFrame con el resumen por cada IDENTIFICACION.
    """
    def construir_resumen(grupo):
        msg = []
        msg.append("TIENE GRADO DE TECNICO" if (grupo["GRADO_TECNICO"] == "SI").any() 
                   else "NO TIENE GRADO DE TECNICO")
        
        msg.append("TIENE GRADO DE TECNOLOGO" if (grupo["GRADO_TECNOLOGO"] == "SI").any() 
                   else "NO TIENE GRADO DE TECNOLOGO")
        
        msg.append("TIENE GRADO DE PROFESIONAL" if (grupo["GRADO_PROFESIONAL"] == "SI").any() 
                   else "NO TIENE GRADO DE PROFESIONAL")
        
        return " , ".join(msg)
    
    return df.groupby("IDENTIFICACION").apply(construir_resumen).reset_index(name="RESUMEN")


In [13]:
# Nueva columna inicialmente en "OK"
df_no_graduados["GRADOS_ANTERIORES"] = "OK"

# Validación para TECNÓLOGO: debe tener TECNICO
df_no_graduados.loc[
    (df_no_graduados["CICLO"] == "TECNOLOGO") & (df_no_graduados["GRADO_TECNICO"] != "SI"),
    "GRADOS_ANTERIORES"
] = "No grado TC"

# Validación para PROFESIONAL: debe tener TECNICO y TECNÓLOGO
df_no_graduados.loc[
    (df_no_graduados["CICLO"] == "PROFESIONAL") & (df_no_graduados["GRADO_TECNICO"] != "SI") & (df_no_graduados["GRADO_TECNOLOGO"] != "SI"),
    "GRADOS_ANTERIORES"
] = "No grado TC y TG"

df_no_graduados.loc[
    (df_no_graduados["CICLO"] == "PROFESIONAL") & (df_no_graduados["GRADO_TECNICO"] != "SI") & (df_no_graduados["GRADO_TECNOLOGO"] == "SI"),
    "GRADOS_ANTERIORES"
] = "No grado TC"

df_no_graduados.loc[
    (df_no_graduados["CICLO"] == "PROFESIONAL") & (df_no_graduados["GRADO_TECNICO"] == "SI") & (df_no_graduados["GRADO_TECNOLOGO"] != "SI"),
    "GRADOS_ANTERIORES"
] = "No grado TG"
df_no_graduados.loc[
    (df_no_graduados["CICLO"] == "PROFESIONAL") & (df_no_graduados["GRADO_TECNICO"] != "SI") & (df_no_graduados["GRADO_TECNOLOGO"] != "SI") & (df_no_graduados["GRADO_PROFESIONAL"] != "SI"),
    "GRADOS_ANTERIORES"
] = "No grado TC, TG y PR"

In [ ]:
df_no_graduados

In [1]:
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
import numpy as np


# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)
   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print("Error:", e)
    
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
           SELECT DISTINCT 
            A.Identificacion, 
            A.Periodo,
            A.ciclo,
            A.Semestre_SINU, 
            A.Programa,
            A.pensum, 
            A.Cod_Unidad,
            A.[PENSAMIENTO CUNISTA]
            FROM SRUBIO.GRADO_OPORTUNO2 AS A
            WHERE [PENSAMIENTO CUNISTA] = 'SIN CURSAR'
              AND Tipo_grado != 'Aun no se debe hacer seguimiento'
                         
                         
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_registros = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

Conexion a la base de datos realizada


In [5]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
            SELECT DISTINCT 
            B.NUM_IDENTIFICACION, 
            CONCAT(B.NOM_TERCERO,' ',B.SEG_NOMBRE,' ',B.PRI_APELLIDO,' ',B.SEG_APELLIDO) AS NOMBRE,
            B.Email_institucional
            from estudiantes_activos AS B
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_activos = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

        

In [6]:
df_activos

,NUM_IDENTIFICACION,NOMBRE,Email_institucional
0,094135594,JAVIER EDUARDO PÉREZ GOYO,javier.perezg@cun.edu.co
1,1000000426,Brayan Fabian Davila Ramirez,brayan.davila@cun.edu.co
2,1000000479,CRISTIAN EDUARDO SALAMANCA GONZALEZ,cristian.salamancaggo@cun.edu.co
3,1000000528,VALERIE CAROLINA PUENTES VELANDIA,valerie.puentes@cun.edu.co
4,1000000551,JOSTIN JESID GARZON CUERVO,jostin.garzon@cun.edu.co
...,...,...,...
64240,9971812,ELIAS ALEXANDER GRISALES ALZATE,elias.grisales@cun.edu.co
64241,9975615,JOHN FREDY VILLA,johnf.villa@cun.edu.co
64242,9975736,LUIS OSWALDO DIAZ BASTIDAS,luis.diazb@cun.edu.co
64243,9976447,ALEXANDER VELÁSQUEZ VILLA,alexander.velasquez@cun.edu.co


In [ ]:
df_registros_2 = df_registros.merge(df_activos, how='inner', left_on='Identificacion', right_on='NUM_IDENTIFICACION')
df_registros_2

,Identificacion,Periodo,ciclo,Semestre_SINU,Programa,pensum,Cod_Unidad,PENSAMIENTO CUNISTA,NUM_IDENTIFICACION,NOMBRE,Email_institucional
0,1000004140,2025C,PROFESIONAL,4.0,DERECHO,2035,1DC01,SIN CURSAR,1000004140,KAREN TATIANA ROSALES TORDECILLA,karen.rosales@cun.edu.co
1,1000005657,25V04,PROFESIONAL,8.0,INGENIERIA DE SISTEMAS,VG001,VP14G,SIN CURSAR,1000005657,JORDAN ESTIBEN ROJAS COTACIO,jordan.rojas@cun.edu.co
2,1000034951,2025A,PROFESIONAL,5.0,DERECHO,2035,1DC01,SIN CURSAR,1000034951,KAREN ANDREA MARTINEZ HERNANDEZ,karen.martinezhhe@cun.edu.co
3,1000048472,2025C,TECNICO,3.0,DISEÑO DE MODAS,5031,1TC57,SIN CURSAR,1000048472,GISELL ESTEFANIA PAREDES BENAVIDES,gisell.paredes@cun.edu.co
4,1000076510,25V03,PROFESIONAL,9.0,INGENIERIA DE SISTEMAS,VG002,VP15G,SIN CURSAR,1000076510,LAURA ANGÉLICA BERNAL ROMERO,laura.bernalrrorom@cun.edu.co
...,...,...,...,...,...,...,...,...,...,...,...
791,80882030,22V01,TECNICO,1.0,ADMINISTRACION DE EMPRESAS,1037,VTE01,SIN CURSAR,80882030,ALVARO GALLEGO SANCHEZ,alvaro.gallego@cun.edu.co
792,92153860,22V01,TECNOLOGO,7.0,INGENIERIA DE SISTEMAS,VG001,VG14G,SIN CURSAR,92153860,JOSE TOMAS VERGARA DIAZ,jose.vergarad@cun.edu.co
793,92276622,2015A,TECNOLOGO,6.0,ADMINISTRACION DE EMPRESAS,1018,GSJ01,SIN CURSAR,92276622,RAFAEL ARMANDO ROMERO OCHOA,rafael.romero@cun.edu.co
794,93392771,2018B,PROFESIONAL,9.0,INGENIERIA ELECTRONICA,2027,1PC02,SIN CURSAR,93392771,JOSE EVER PEÑA VASQUEZ,Jose.Penav@cun.edu.co


In [ ]:

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)
   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print("Error:", e)
    
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
gyu          v
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_materias = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

Conexion a la base de datos realizada


In [24]:
df_materias

,COD_MATERIA,COD_PERIODO,NOM_MATERIA,COD_UNIDAD,COD_PENSUM
0,56810,2013A,PENSAMIENTO SOCIOLOGICO,PBC05,1023
1,56810,2013A,PENSAMIENTO SOCIOLOGICO,TBC05,1023
2,56810,2013A,PENSAMIENTO SOCIOLOGICO,TBV04,1023
3,56810,2013A,PENSAMIENTO SOCIOLOGICO,TES04,1023
4,56810,2013A,PENSAMIENTO SOCIOLOGICO,TGR04,1023
...,...,...,...,...,...
27841,TIF04,25V02,CREATIVIDAD Y PENSAMIENTO INNOVADOR EN CIENCIA...,VTM14,6017
27842,TIF04,25V03,CREATIVIDAD Y PENSAMIENTO INNOVADOR EN CIENCIA...,VTM14,6017
27843,TIF04,25V04,CREATIVIDAD Y PENSAMIENTO INNOVADOR EN CIENCIA...,TVM06,VTG12
27844,TIF04,25V04,CREATIVIDAD Y PENSAMIENTO INNOVADOR EN CIENCIA...,VTM14,6017


In [27]:
df_total = df_registros_2.merge(df_materias, how='inner',left_on=['Periodo','pensum','Cod_Unidad'], right_on=['COD_PERIODO','COD_PENSUM','COD_UNIDAD'])
df_total.drop_duplicates(inplace=True)
df_total

,Identificacion,Periodo,ciclo,Semestre_SINU,Programa,pensum,Cod_Unidad,PENSAMIENTO CUNISTA,NUM_IDENTIFICACION,NOMBRE,Email_institucional,COD_MATERIA,COD_PERIODO,NOM_MATERIA,COD_UNIDAD,COD_PENSUM
0,1000005657,25V04,PROFESIONAL,8.0,INGENIERIA DE SISTEMAS,VG001,VP14G,SIN CURSAR,1000005657,JORDAN ESTIBEN ROJAS COTACIO,jordan.rojas@cun.edu.co,56946,25V04,PENSAMIENTO CUNISTA 2,VP14G,VG001
1,1000005657,25V04,PROFESIONAL,8.0,INGENIERIA DE SISTEMAS,VG001,VP14G,SIN CURSAR,1000005657,JORDAN ESTIBEN ROJAS COTACIO,jordan.rojas@cun.edu.co,56947,25V04,PENSAMIENTO CUNISTA 3,VP14G,VG001
2,1000076510,25V03,PROFESIONAL,9.0,INGENIERIA DE SISTEMAS,VG002,VP15G,SIN CURSAR,1000076510,LAURA ANGÉLICA BERNAL ROMERO,laura.bernalrrorom@cun.edu.co,TI003,25V03,CATEDRA DE PENSAMIENTO CUNISTA III,VP15G,VG002
3,1000136184,25V03,PROFESIONAL,7.0,ADMINISTRACION DE EMPRESAS,1037,PVC06,SIN CURSAR,1000136184,SHARON NICOLL NOREÑA SANCHEZ,sharon.norena@cun.edu.co,TI001,25V03,CATEDRA DE PENSAMIENTO CUNISTA I,PVC06,1037
4,1000136184,25V03,PROFESIONAL,7.0,ADMINISTRACION DE EMPRESAS,1037,PVC06,SIN CURSAR,1000136184,SHARON NICOLL NOREÑA SANCHEZ,sharon.norena@cun.edu.co,TI003,25V03,CATEDRA DE PENSAMIENTO CUNISTA III,PVC06,1037
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1092,92153860,22V01,TECNOLOGO,7.0,INGENIERIA DE SISTEMAS,VG001,VG14G,SIN CURSAR,92153860,JOSE TOMAS VERGARA DIAZ,jose.vergarad@cun.edu.co,56946,22V01,PENSAMIENTO CUNISTA 2,VG14G,VG001
1093,92153860,22V01,TECNOLOGO,7.0,INGENIERIA DE SISTEMAS,VG001,VG14G,SIN CURSAR,92153860,JOSE TOMAS VERGARA DIAZ,jose.vergarad@cun.edu.co,56947,22V01,PENSAMIENTO CUNISTA 3,VG14G,VG001
1094,92276622,2015A,TECNOLOGO,6.0,ADMINISTRACION DE EMPRESAS,1018,GSJ01,SIN CURSAR,92276622,RAFAEL ARMANDO ROMERO OCHOA,rafael.romero@cun.edu.co,56948,2015A,PENSAMIENTO ECONOMICO,GSJ01,1018
1095,92276622,2015A,TECNOLOGO,6.0,ADMINISTRACION DE EMPRESAS,1018,GSJ01,SIN CURSAR,92276622,RAFAEL ARMANDO ROMERO OCHOA,rafael.romero@cun.edu.co,93755,2015A,PENSAMIENTO LOGICO,GSJ01,1018


In [19]:
df_total.columns

Index(['Identificacion', 'Periodo', 'ciclo', 'Semestre_SINU', 'Programa',
       'pensum', 'Cod_Unidad', 'PENSAMIENTO CUNISTA', 'NUM_IDENTIFICACION',
       'NOMBRE', 'Email_institucional', 'COD_MATERIA', 'COD_PERIODO',
       'NOM_MATERIA', 'COD_UNIDAD', 'COD_PENSUM'],
      dtype='object')

In [26]:
df_total_2 = df_total[['Identificacion','NOMBRE','Email_institucional', 'Periodo', 'ciclo', 'Semestre_SINU', 'Programa',
       'pensum', 'Cod_Unidad', 'PENSAMIENTO CUNISTA','COD_MATERIA', 'NOM_MATERIA', ]]
df_total_2

,Identificacion,NOMBRE,Email_institucional,Periodo,ciclo,Semestre_SINU,Programa,pensum,Cod_Unidad,PENSAMIENTO CUNISTA,COD_MATERIA,NOM_MATERIA
0,1000004140,KAREN TATIANA ROSALES TORDECILLA,karen.rosales@cun.edu.co,2025C,PROFESIONAL,4.0,DERECHO,2035,1DC01,SIN CURSAR,NaN,NaN
1,1000005657,JORDAN ESTIBEN ROJAS COTACIO,jordan.rojas@cun.edu.co,25V04,PROFESIONAL,8.0,INGENIERIA DE SISTEMAS,VG001,VP14G,SIN CURSAR,56946,PENSAMIENTO CUNISTA 2
2,1000005657,JORDAN ESTIBEN ROJAS COTACIO,jordan.rojas@cun.edu.co,25V04,PROFESIONAL,8.0,INGENIERIA DE SISTEMAS,VG001,VP14G,SIN CURSAR,56947,PENSAMIENTO CUNISTA 3
3,1000034951,KAREN ANDREA MARTINEZ HERNANDEZ,karen.martinezhhe@cun.edu.co,2025A,PROFESIONAL,5.0,DERECHO,2035,1DC01,SIN CURSAR,NaN,NaN
4,1000048472,GISELL ESTEFANIA PAREDES BENAVIDES,gisell.paredes@cun.edu.co,2025C,TECNICO,3.0,DISEÑO DE MODAS,5031,1TC57,SIN CURSAR,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1460,92153860,JOSE TOMAS VERGARA DIAZ,jose.vergarad@cun.edu.co,22V01,TECNOLOGO,7.0,INGENIERIA DE SISTEMAS,VG001,VG14G,SIN CURSAR,56947,PENSAMIENTO CUNISTA 3
1461,92276622,RAFAEL ARMANDO ROMERO OCHOA,rafael.romero@cun.edu.co,2015A,TECNOLOGO,6.0,ADMINISTRACION DE EMPRESAS,1018,GSJ01,SIN CURSAR,56948,PENSAMIENTO ECONOMICO
1462,92276622,RAFAEL ARMANDO ROMERO OCHOA,rafael.romero@cun.edu.co,2015A,TECNOLOGO,6.0,ADMINISTRACION DE EMPRESAS,1018,GSJ01,SIN CURSAR,93755,PENSAMIENTO LOGICO
1463,93392771,JOSE EVER PEÑA VASQUEZ,Jose.Penav@cun.edu.co,2018B,PROFESIONAL,9.0,INGENIERIA ELECTRONICA,2027,1PC02,SIN CURSAR,NaN,NaN
